In [ ]:
import geopandas as gpd
from odc.stac import load
from pystac.client import Client
from pathlib import Path
import pandas as pd
from src.utils import make_indices, mask_land, mask_deeps, S2_BANDS

import warnings
warnings.filterwarnings("ignore")

In [ ]:
MASK_DEEP_WATER = False
MASK_LAND = True

# Open the regions file
regions = gpd.read_file('postcards.geojson')

# List the geopackage files in data
data_dir = Path('data')

In [ ]:
# Configure our STAC API
client = Client.open("https://stac.digitalearthpacific.org")
collection = "dep_s2_geomad"
year = "2024"

In [ ]:
if MASK_LAND:
    print("Masking land")

if MASK_DEEP_WATER:
    print("Masking deep water\n")

for region in regions.itertuples():
    file = data_dir / f"{region.name}_20.gpkg"
    is_atoll = region.type == "Atoll"

    # Load the file
    region_df = gpd.read_file(file)
    print(f"Working on {region.name}, which has {len(region_df)} points")

    # Get total bounds in lat/lon
    bbox = region_df.to_crs("epsg:4326").total_bounds

    # Get geomad data for the aoi
    items = client.search(
        collections=[collection],
        bbox=bbox,
        datetime=year,
    ).item_collection()
    print(f"Found {len(items)} GeoMAD items for {region.name}")

    s2_bands_no_scl = [
        band for band in S2_BANDS if band != "scl"
    ]

    geomad = load(
        items,
        bbox=bbox,
        chunks={"x": 512, "y": 512},
        nodata=0,
        measurements=s2_bands_no_scl,
    )

    # NOTE: GEBCO is not good enough for masking at our scale
    # # Mask with GEBCO
    # if not is_atoll:
    #     geomad = mask_with_gebco(geomad, GEBCO_DEPTH_LIMIT)

    # Create some indices to help with the classification
    geomad = make_indices(geomad)

    if MASK_LAND:
        geomad = mask_land(geomad)

    if MASK_DEEP_WATER:
        geomad = mask_deeps(geomad)

    # Combine the geomad values with the region points
    region_da = region_df.assign(
        x=region_df.geometry.x, y=region_df.geometry.y
    ).to_xarray()
    training_values = (
        geomad.sel(region_da[["x", "y"]], method="nearest")
        .squeeze()
        .compute()
        .to_pandas()
    )

    # Merge the original depth column with the new values
    training_df = pd.concat([region_df["depth"], training_values], axis=1)

    # Drop nans
    training_df = training_df.dropna()

    # Drop non-needed columns
    training_df = training_df.drop(columns=["spatial_ref", "time"])
    out_csv = data_dir / f"training/{region.name}_land_mask.csv"

    print(f"Finished {region.name}, with {len(training_df)} rows. Saving to {out_csv}\n-")

    training_df.to_csv(out_csv, index=False)

print("Finished all regions")